# Faz 3 - Baseline Modeller

Persistence, mevsimsel persistence, Doğrusal Regresyon ve XGBoost; hepsi aynı test penceresi kümesinde ve tek `evaluate()` ile ölçülmüştür. Bu notebook eğitimi tekrarlamaz; `results/` altındaki kayıtlı çıktıları okur.

In [1]:
import sys, json, subprocess
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd
from src import config
config.apply_plot_style()
print(config.summary())

Yıllar: 2015-2016
Ham parquet: C:\Users\user\Desktop\merve\data\raw\pvdaq_sys4_2015_2016.parquet
Bölme: train 1 segment, val 6 segment, test 6 segment; veri sonu 2016-12-13 00:00
Hedef: ac_power [W]
Kapasite: 1.0 kW = 1000.0 W (nRMSE paydası: CAPACITY_W)


## 1. Baseline tanımları
- **Persistence:** tahmin ufkundaki her adım için son gözlenen AC güç değeri.
- **Mevsimsel persistence:** dün aynı saatin gözlemi (pencere indeksi h-1).
- **Gece kuralı:** hedef zamanında güneş yüksekliği <= 0 ise tahmin 0 yapılır. Bu, deterministik astronomik bilgidir ve tüm modellere aynı şekilde uygulanır.
- **Doğrusal Regresyon ve XGBoost:** 16 adım çıktıyı tablo öznitelikleri üzerinden tahmin eder.

## 2. Öznitelikler ve XGBoost ayarları

In [2]:
info = json.load(open(config.MODELS_DIR / 'baselines_info.json', encoding='utf-8'))
names = info['tabular_feature_names']
print(len(names), 'öznitelik')
print(names)
print('XGBoost:', info['xgboost'])
print('LinReg:', info['linreg'])

57 öznitelik
['ac_power_lag0', 'ac_power_lag1', 'ac_power_lag2', 'ac_power_lag3', 'ac_power_lag4', 'ac_power_lag5', 'ac_power_lag6', 'ac_power_lag7', 'poa_irradiance_lag0', 'poa_irradiance_lag1', 'poa_irradiance_lag2', 'poa_irradiance_lag3', 'poa_irradiance_lag4', 'poa_irradiance_lag5', 'poa_irradiance_lag6', 'poa_irradiance_lag7', 'ac_power_dun_h1', 'ac_power_dun_h2', 'ac_power_dun_h3', 'ac_power_dun_h4', 'ac_power_dun_h5', 'ac_power_dun_h6', 'ac_power_dun_h7', 'ac_power_dun_h8', 'ac_power_dun_h9', 'ac_power_dun_h10', 'ac_power_dun_h11', 'ac_power_dun_h12', 'ac_power_dun_h13', 'ac_power_dun_h14', 'ac_power_dun_h15', 'ac_power_dun_h16', 'poa_irradiance_dun_h1', 'poa_irradiance_dun_h2', 'poa_irradiance_dun_h3', 'poa_irradiance_dun_h4', 'poa_irradiance_dun_h5', 'poa_irradiance_dun_h6', 'poa_irradiance_dun_h7', 'poa_irradiance_dun_h8', 'poa_irradiance_dun_h9', 'poa_irradiance_dun_h10', 'poa_irradiance_dun_h11', 'poa_irradiance_dun_h12', 'poa_irradiance_dun_h13', 'poa_irradiance_dun_h14', 

## 3. Metrikler
nRMSE = RMSE / kurulu kapasite (`config.CAPACITY_W`); kW sütunları W değerlerinin /1000 halidir. Birincil metrik gündüz-yalnız hesaplanır (gece tahmini kural gereği 0'dır, 'tüm saat' metrikleri gece örnekleriyle sulanır).

In [3]:
print('capacity_W =', config.CAPACITY_W)
a = pd.read_csv(config.METRICS_DIR / 'all_metrics.csv')
b = a[a['model'].isin(['persistence','seasonal_persistence','linreg','xgboost'])]
print(b.pivot_table(index='model', columns=['mask','scope'], values='RMSE_W').round(1).to_string())
print()
print(b.pivot_table(index='model', columns=['mask','scope'], values='nRMSE').round(4).to_string())
print()
print(b[(b['mask']=='day')&(b['scope']=='h4')][['model','MAE_kW','RMSE_kW','R2']].round(4).to_string(index=False))

capacity_W = 1000.0
mask                    all           day       
scope                    h4 mean16     h4 mean16
model                                           
linreg                103.8  120.7  148.2  172.2
persistence           138.8  216.9  197.2  304.6
seasonal_persistence  171.9  171.9  246.9  246.9
xgboost                93.5  108.3  134.0  155.2

mask                     all             day        
scope                     h4  mean16      h4  mean16
model                                               
linreg                0.1038  0.1207  0.1482  0.1722
persistence           0.1388  0.2169  0.1972  0.3046
seasonal_persistence  0.1719  0.1719  0.2469  0.2469
xgboost               0.0935  0.1083  0.1340  0.1552

               model  MAE_kW  RMSE_kW     R2
         persistence  0.1468   0.1972 0.5379
seasonal_persistence  0.1524   0.2469 0.2759
              linreg  0.1057   0.1482 0.7390
             xgboost  0.0895   0.1340 0.7867


## 4. Birim testleri

In [4]:
for t in ('test_evaluate.py', 'test_baselines.py'):
    r = subprocess.run([sys.executable, '-I', str(PROJECT_ROOT/'tests'/t)], capture_output=True, text=True, encoding='utf-8', cwd=PROJECT_ROOT)
    print(t); print(r.stdout); print(r.stderr[-500:])

test_evaluate.py
OK test_metrik_elle
OK test_gece_kurali
OK test_bos_maske_ve_sabit_hedef
OK test_evaluate_satirlari
OK test_summarize
OK test_capacity_none
TESTLER_TAMAM




test_baselines.py
OK test_seasonal_indeks
OK test_persistence
OK test_tabular_sekil
OK test_kanal_adla
TESTLER_TAMAM


